In [1]:
# | default_exp seo_report


In [2]:
# | export
from sqlmodel import Session, select
from seo_rat.article import get_article_by_path, insert_article, Article
from seo_rat.sqlite_db import get_session
from seo_rat.models import Website
from seo_rat.content_mapper import map_all_urls_to_files, fetch_sitemap_urls
from seo_rat.content_parser import (
    parse_metadata,
    parse_notebook_metadata,
    remove_metadata,
    extract_headers,
    extract_images,
    extract_links,
    filter_external_links,
    filter_internal_links,
    check_desc_length,
    check_content_length,
    check_title_length,
    imgs_missing_alts,
    extract_notebook_content, extract_html_metadata, fetch_url_as_markdown,
)
from seo_rat.seo_content_analysis import check_h1_count, check_keyword_placement
from seo_rat.gsc_insights import detect_query_trends, classify_page_intents, find_green_keywords
import httpx

FETCH = "::fetch::"



In [3]:
# | export
def sync_articles_to_db(session: Session,  # Active database session
                        website_id: int,  # Parent website ID
                        url_file_mapping: dict[str, str]  # URL → file path mapping
                        ) -> None:
    "Insert articles into the database, skipping already existing ones."
    for url, file_path in url_file_mapping.items():
        if not file_path: continue
        if file_path == FETCH:
            existing = session.exec(select(Article).where(Article.url == url)).first()
        else:
            existing = get_article_by_path(session, file_path)
        if not existing:
            insert_article(session, Article(website_id=website_id, file_path=file_path, url=url))


In [5]:
# | hide
#| eval: false
from dotenv import load_dotenv
import os

load_dotenv()
# BASE_PATH = os.environ["SEO_RAT_BASE_PATH"]
BASE_PATH = os.environ["AWAZLY_PATH"]
# BASE_PATH = os.environ["emdadelgaz_PATH"]
# BASE_PATH = os.environ["EMDAD_PATH"]
# BASE_PATH = "/home/kobo/Desktop/astro_hacking/ainclean"

In [6]:

#| eval: false
#| hide
from pathlib import Path


def get_url_file_mapping_ain() -> dict[str, str]:
    urls = fetch_sitemap_urls("https://www.alainclean.com/sitemap.xml")
    mapping = map_all_urls_to_files(
        base_path="/home/kobo/Desktop/astro_hacking/ainclean/articles",
        site_url="https://www.alainclean.com/article",
        urls=[u for u in urls if "/article/" in u],
        mode="direct"
    )

    # mark unmapped URLs for fetching
    for url in urls:
        if url not in mapping:
            mapping[url] = FETCH
    return mapping


def get_url_file_mapping_smaa() -> dict[str, str]:
    from urllib.parse import urlparse, parse_qs, unquote
    base = "/home/kobo/Desktop/astro_hacking/samma_garden/articles"
    urls = fetch_sitemap_urls("https://www.smaagarden.com/sitemap.xml")
    mapping = {}
    for url in urls:
        qs = parse_qs(urlparse(url).query)
        if slug := qs.get("slug", [None])[0]:
            path = f"{base}/{unquote(slug)}.md"
            mapping[url] = path if Path(path).exists() else FETCH
        else:
            mapping[url] = FETCH
    return mapping


def get_url_file_mapping_gpuvec() -> dict[str, str]:
    from seo_rat.content_mapper import map_all_urls_to_files
    from seo_rat.index_tracking import fetch_sitemap_urls
    from seo_rat.seo_report import FETCH

    urls = fetch_sitemap_urls("https://gpuvec.com/sitemap.xml")

    mapping = map_all_urls_to_files(
        base_path="/home/kobo/Desktop/Desktop/gpuvec",
        site_url="https://gpuvec.com",
        urls=urls,
        mode="direct"
    )

    for url in urls:
        if url not in mapping:
            mapping[url] = FETCH
    return mapping




In [ ]:

#| eval: false
#| hide
with get_session() as session:
    websites = session.exec(select(Website)).all()
    for w in websites:
        print(f"ID: {w.id}, URL: {w.url}")

    urls = fetch_sitemap_urls("https://kareemai.com/sitemap.xml")
    # url_mapping = map_all_urls_to_files(
    #     base_path=f"{BASE_PATH}",
    #     site_url="https://www.smaagarden.com",
    #     urls=urls,
    #     mode="direct",
    # )
    url_mapping = map_all_urls_to_files(
        base_path=f"{BASE_PATH}",
        site_url="https://kareemai.com/",
        urls=urls,
        mode="direct"
    )
    sync_articles_to_db(session, website_id=4, url_file_mapping=url_mapping)
    articles = session.exec(select(Article).where(Article.website_id == 4)).all()
    for article in articles:
        print(article)


ID: 1, URL: https://awazly.com
ID: 2, URL: https://shelid.com
ID: 3, URL: https://emdadelgaz.com
ID: 4, URL: https://kareemai.com
ID: 6, URL: https://www.smaagarden.com/
ID: 7, URL: https://test-example.com
ID: 8, URL: https://gpuvec.com/
ID: 9, URL: https://alainclean.com/


# SEO Report
> Sync articles, detect duplicate metadata, and generate SEO reports from GSC data.

## Delete Articles in wrong Website

In [7]:
#| hide
#| eval: false
# from sqlmodel import delete
# session.exec(
#     delete(Article).where(
#         Article.website_id == 2,
#         Article.file_path.contains("emdadgaz")
#     )
# )
# session.commit()


In [8]:
# | export

def _collect_metadata_field(session: Session,  # Active database session
                            website_id: int,  # Parent website ID
                            field: str  # Metadata field to extract
                            ) -> dict[str, str]:
    "Collect metadata field values from all articles in a website."
    from seo_rat.content_parser import parse_metadata, parse_notebook_metadata
    values = {}
    for article in get_articles_by_website(session, website_id):
        if article.file_path == FETCH:
            continue
        content = open(article.file_path).read()
        meta = parse_notebook_metadata(content) if article.file_path.endswith(".ipynb") else parse_metadata(content)
        values[article.file_path] = meta.get(field, "")
    return values


def find_duplicate_metadata(session: Session,  # Active database session
                            field: str,  # Metadata field to check
                            website_id: int,  # Parent website ID
                            similarity_threshold: float = 0.9  # Minimum similarity to flag
                            ) -> list[dict]:
    "Find articles with duplicate or very similar metadata field values."
    from seo_rat.content_parser import calculate_similarity
    values = _collect_metadata_field(session, website_id, field)
    duplicates = []
    checked = set()
    for path1, field1 in values.items():
        for path2, field2 in values.items():
            if path1 >= path2 or (path1, path2) in checked: continue
            if not field1 or not field2: continue
            if (similarity := calculate_similarity(field1, field2)) >= similarity_threshold:
                duplicates.append({"file1": path1, "file2": path2,
                                   f"{field}_1": field1, f"{field}_2": field2,
                                   "similarity": similarity})
            checked.add((path1, path2))
    return duplicates

In [11]:
#| hide
#| eval: false
from seo_rat.models import Website
from seo_rat.gsc_storage import GSCAnalytics, func

session.exec(select(Website)).all()
session.exec(
    select(GSCAnalytics.site_url, func.count().label("rows"))
    .group_by(GSCAnalytics.site_url)
).all()


[('sc-domain:alainclean.com', 4738),
 ('sc-domain:awazly.com', 248061),
 ('sc-domain:emdadelgaz.com', 147806),
 ('sc-domain:gpuvec.com', 128604),
 ('sc-domain:kareemai.com', 96939),
 ('sc-domain:shelid.com', 166540),
 ('sc-domain:smaagarden.com', 3598)]

In [13]:
# | hide
#| eval: false
website = Website(url="https://alainclean.com/", name="alainclean", lang="ar",
                  desc="موقع متخصص في خدمات تنظيف املنازل في العين والإمارات")

with get_session() as session:
    session.add(website)
    session.commit()
    session.refresh(website)
    print(website.id)


IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: website.url
[SQL: INSERT INTO website (url, name, "desc", lang, created_at, site_type, content_dir) VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: ('https://alainclean.com/', 'alainclean', 'موقع متخصص في خدمات تنظيف املنازل في العين والإمارات', 'ar', '2026-05-05 14:25:26.954605', None, '')]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [12]:
#| hide
#| eval: false
from seo_rat.models import get_all_websites, print_websites

with get_session() as session:
    websites = get_all_websites(session)
    print_websites(websites)

                           🌐 Websites                           
┏━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┓
┃ ID ┃ Name       ┃ URL                         ┃ Type   ┃ Lang ┃
┡━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━┩
│  1 │ Awazly     │ https://awazly.com          │ astro  │  ar  │
│  2 │ shelid     │ https://shelid.com          │ astro  │  ar  │
│  3 │ emdadelgaz │ https://emdadelgaz.com      │ astro  │  ar  │
│  4 │ kareemai   │ https://kareemai.com        │ quarto │  ar  │
│  6 │ smaagarden │ https://www.smaagarden.com/ │        │  ar  │
│  7 │ Test Site  │ https://test-example.com    │ astro  │  en  │
│  8 │ gpuvec     │ https://gpuvec.com/         │        │  en  │
│  9 │ alainclean │ https://alainclean.com/     │        │  ar  │
└────┴────────────┴─────────────────────────────┴────────┴──────┘

## Make FocusKeyword to be Top query from GSC


In [13]:
#| export 
from seo_rat.gsc_storage import get_top_queries
from seo_rat.article import get_articles_by_website
from seo_rat.gsc_client import get_date_range
from seo_rat.seo_content_analysis import check_keyword_placement, content_freshness
from seo_rat.gsc_insights import find_missing_queries

In [14]:
from seo_rat.gsc_storage import sync_missing_dates
# auth.authenticate()
#| hide
#| eval: false
from seo_rat.gsc_client import GSCAuth
from seo_rat.gsc_storage import daily_sync

start, end = get_date_range("last_days", days=31 * 4)
print(start, end)
auth = GSCAuth(secrets_file="client_secrets.json")
with get_session() as session:
    sync_missing_dates(session, auth, site_url="sc-domain:kareemai.com", start_date=start, end_date=end)


2026-01-02 2026-05-06
Synced 2026-05-03: 413 records
Synced 2026-05-04: 472 records
Synced 2026-05-05: 421 records
Synced 2026-05-06: 442 records


In [16]:
#| hide
#| eval: false
website_name = "https://awazly.com/"

with get_session() as session:
    articles = get_articles_by_website(session, website_id=1)
    print(len(articles))
    for article in articles:
        page_path = article.url.removeprefix(website_name)
        print(page_path)
        print(article)
        top_query = get_top_queries(session=session, site_url="sc-domain:kareemai.com", start_date=start,
                                    end_date=end, page_path=page_path, limit=1, sort_by="impressions")
        top_5_query = get_top_queries(session=session, site_url="sc-domain:kareemai.com", start_date=start,
                                      end_date=end, page_path=page_path, limit=5, sort_by="impressions")
        try:
            print(top_query[0]['query'])
            article.focus_keyword = top_query[0]['query']
            article.secondary_keywords = [q["query"] for q in top_5_query[:5]]
            session.add(article)
            session.commit()
        except IndexError:
            print("No queries found for this page")


87
malm-dhanat-aldwadmy
url='https://awazly.com/malm-dhanat-aldwadmy' id=1 secondary_keywords=['طريقة تنظيف سيراميك الباركيه من البويه', 'عامل بويه', 'معلم تركيب باركيه بالمدينة المنورة', 'عمائر في الدوادمي', 'عمائر تجارية في الدوادمي'] last_optimized=None focus_keyword='طريقة تنظيف سيراميك الباركيه من البويه' website_id=1 file_path='/home/kobo/Desktop/astro_hacking/awazl/src/content/post/معلم_دهانات_الدوادمي.md' target_goal=None created_at=datetime.datetime(2026, 3, 29, 1, 29, 26, 457544)
No queries found for this page
malm-dhanat-kfr-alshykh
url='https://awazly.com/malm-dhanat-kfr-alshykh' id=2 secondary_keywords=['معلم دهانات', 'افضل معلم دهانات', 'معلم دهان', 'عيوب دهان ستوكو', 'محل دهانات قريب مني'] last_optimized=None focus_keyword='معلم دهانات' website_id=1 file_path='/home/kobo/Desktop/astro_hacking/awazl/src/content/post/معلم_دهانات_كفر_الشيخ.md' target_goal=None created_at=datetime.datetime(2026, 3, 29, 1, 29, 26, 469077)
No queries found for this page
ghsyl-knb-baldwadmy
url

In [17]:
# | export
def analyze_links(content: str,  # Page content
                  domain: str  # Site domain to classify links
                  ) -> dict:
    "Analyze internal and external links in content."
    all_urls = list(extract_links(content).keys())
    return {"total_links": len(all_urls),
            "internal_count": len(filter_internal_links(all_urls, domain)),
            "external_count": len(filter_external_links(all_urls, domain)),
            "internal_links": filter_internal_links(all_urls, domain),
            "external_links": filter_external_links(all_urls, domain)}


In [18]:
# | hide
#| eval: false

from seo_rat.content_parser import remove_metadata

with get_session() as session:
    article = session.exec(select(Article)).first()
    print(article)
    with open(article.file_path, "r") as f:
        content = remove_metadata(f.read())

    link_analysis = analyze_links(content, "gpuvec.com")
    print(link_analysis)

    print(f"Total: {link_analysis['total_links']}")
    print(f"Internal: {link_analysis['internal_count']}")
    print(f"External: {link_analysis['external_count']}")


url='https://awazly.com/malm-dhanat-aldwadmy' id=1 secondary_keywords=['طريقة تنظيف سيراميك الباركيه من البويه', 'عامل بويه', 'معلم تركيب باركيه بالمدينة المنورة', 'عمائر في الدوادمي', 'عمائر تجارية في الدوادمي'] last_optimized=None focus_keyword='طريقة تنظيف سيراميك الباركيه من البويه' website_id=1 file_path='/home/kobo/Desktop/astro_hacking/awazl/src/content/post/معلم_دهانات_الدوادمي.md' target_goal=None created_at=datetime.datetime(2026, 3, 29, 1, 29, 26, 457544)
{'total_links': 5, 'internal_count': 0, 'external_count': 5, 'internal_links': [], 'external_links': ['https://awazly.com/trkyb-aybwksy-almdynh-almnwrh', 'https://awazly.com/trkyb-aybwksy-bmkh', 'https://awazly.com/trkyb-aybwksy-jdh', 'https://awazly.com/shrkh-azl-aybwksy-ardhyat-balryadh', 'https://awazly.com/malm-dhanat-balryadh']}
Total: 5
Internal: 0
External: 5


In [19]:
# | export
def check_heading_structure(headers: list[dict]  # Headers from `extract_headers`
                            ) -> dict:
    "Check heading structure for H2 presence and skipped levels."
    levels = [int(h["type"][1]) for h in headers]
    skipped = [{"from": headers[i], "to": headers[i + 1]}
               for i in range(len(levels) - 1) if levels[i + 1] - levels[i] > 1]
    return {"has_h2": any(h["type"] == "h2" for h in headers), "skipped_levels": skipped}


In [20]:
# | export

def analyze_article(article: Article,
                    domain: str,
                    is_quarto: bool,
                    title_is_h1: bool = False,
                    desc_field: str = "description",
                    title_field: str = "title",
                    date_field: str = "date",
                    fetch_base_url: str | None = None  # ADD
                    ) -> dict:
    if article.file_path == FETCH:
        fetch_url = article.url.replace(f"https://{domain}", fetch_base_url) if fetch_base_url else article.url
        html = httpx.get(fetch_url).text  # FIX: use fetch_url
        metadata = extract_html_metadata(html)
        content = fetch_url_as_markdown(fetch_url)
        headers = extract_headers(content)
    elif article.file_path.endswith(".ipynb"):
        raw = open(article.file_path).read()
        metadata = parse_notebook_metadata(raw)
        content = extract_notebook_content(raw, is_quarto=is_quarto)
        headers = extract_headers(content)
    else:
        raw = open(article.file_path).read()
        metadata = parse_metadata(raw)
        content = remove_metadata(raw)
        headers = extract_headers(content)

    return {"file_path": article.file_path,
            "title_check": check_title_length(metadata.get(title_field, "")),
            "description_check": check_desc_length(metadata.get(desc_field, "")),
            "content_check": check_content_length(content),
            "h1_check": check_h1_count(headers, title=metadata.get("title", ""), title_is_h1=title_is_h1),
            "missing_alts": imgs_missing_alts(extract_images(content)),
            "link_analysis": analyze_links(content, domain),
            "keyword_placement": check_keyword_placement(article.focus_keyword, metadata=metadata,
                                                         headers=headers, content=content, url=article.url,
                                                         desc_field=desc_field, title_is_h1=title_is_h1),
            "content_freshness": content_freshness(metadata.get(date_field, "")),
            "content_structure": check_heading_structure(headers)}


In [21]:
#| hide
#|eval: false
article = Article(website_id=4, file_path="::fetch::", url="https://kareemai.com")
analyze_article(article, "kareemai.com", is_quarto=True)


{'file_path': '::fetch::',
 'title_check': {'length': 72, 'status': 'too_long', 'min': 50, 'max': 60},
 'description_check': {'length': 148,
  'status': 'too_short',
  'min': 150,
  'max': 160},
 'content_check': {'word_count': 303, 'is_sufficient': True},
 'h1_check': {'h1_count': 1, 'has_single_h1': True},
 'missing_alts': [],
 'link_analysis': {'total_links': 0,
  'internal_count': 0,
  'external_count': 0,
  'internal_links': [],
  'external_links': []},
 'keyword_placement': {'in_h1': False,
  'in_url': False,
  'keyword_in_meta': {'in_title': False, 'in_description': False},
  'keyword_at_first': False},
 'content_freshness': {'last_updated': '',
  'days_since_update': None,
  'is_fresh': True},
 'content_structure': {'has_h2': True, 'skipped_levels': []}}

In [22]:
# | export
def _collect_page_issues(
        report: dict,  # Result from `analyze_article`
        article: Article,  # Article being analyzed
) -> list[str]:
    "Collect all SEO issues from an article report as a list of strings."
    issues = []
    if not report["h1_check"]["has_single_h1"]:
        issues.append("Multiple or no H1")
    if not report["content_check"]["is_sufficient"]:
        issues.append("Content too short")
    if report["link_analysis"]["internal_count"] == 0:
        issues.append("No internal links, at least 3 recommended")
    if report["link_analysis"]["external_count"] == 0:
        issues.append("No external links")
    if report["missing_alts"]:
        issues.append(f"Images missing alt text ({len(report['missing_alts'])})")
    if not report["content_structure"]["has_h2"]:
        issues.append("No H2 headings, at least one recommended")
    for skip in report["content_structure"]["skipped_levels"]:
        issues.append(
            f"Skipped heading: {skip['from']['type']} → {skip['to']['type']} (line {skip['to']['line_number']})"
        )
    tc = report["title_check"]
    if tc["status"] != "optimal":
        issues.append(
            f"Title {tc['status']} ({tc['length']} chars, range {tc['min']}-{tc['max']})"
        )
    dc = report["description_check"]
    if dc["status"] != "optimal":
        issues.append(
            f"Description {dc['status']} ({dc['length']} chars, range {dc['min']}-{dc['max']})"
        )
    if not report["content_freshness"]["is_fresh"]:
        issues.append(
            f"Stale content ({report['content_freshness']['days_since_update']} days old)"
        )
    if article.focus_keyword:
        if not report["keyword_placement"]["in_h1"]:
            issues.append("Focus keyword not in H1")
        if not report["keyword_placement"]["keyword_in_meta"]["in_description"]:
            issues.append("Focus keyword not in description")
        if not report["keyword_placement"]["keyword_in_meta"]["in_title"]:
            issues.append("Focus keyword not in title")
        if not report["keyword_placement"]["keyword_at_first"]:
            issues.append("Focus keyword not in first 100 words")
    return issues


def generate_seo_report(
        session: Session,
        website_id: int,
        domain: str,
        is_quarto: bool,
        title_is_h1: bool = False,
        desc_field: str = "description",
        title_field: str = "title",
        date_field: str = "date",
        include_insights: bool = False,  # Include query intents, trends, green keywords
        days: int = 90,  # Days for insights lookback
        query_limit: int = 100,  # Max queries for insights
        fetch_base_url: str | None = None,
) -> dict:
    "Generate a comprehensive SEO report for all articles in a website."
    articles = get_articles_by_website(session, website_id)
    start, end = get_date_range("last_days", days=90 * 8)
    page_reports, issues = [], {}

    for article in articles:
        try:
            report = analyze_article(
                article,
                domain,
                is_quarto,
                title_is_h1=title_is_h1,
                desc_field=desc_field,
                title_field=title_field,
                date_field=date_field,
                fetch_base_url=fetch_base_url,
            )
            queries = get_top_queries(
                session=session,
                site_url=f"sc-domain:{domain}",
                start_date=start,
                end_date=end,
                page_path=article.url.removeprefix(f"https://{domain}/"),
                limit=500,
                sort_by="impressions",
            )
            if article.file_path == FETCH:
                content = fetch_url_as_markdown(
                    article.url.replace(f"https://{domain}", fetch_base_url)
                    if fetch_base_url
                    else article.url
                )
            else:
                content = remove_metadata(open(article.file_path).read())
            missing_queries = find_missing_queries(queries=queries, content=content)

            missing_queries = find_missing_queries(queries=queries, content=content)
            page_issues = _collect_page_issues(report, article)

            if include_insights:
                page_path = article.url.removeprefix(f"https://{domain}/")
                insight_start, insight_end = get_date_range("last_days", days=days)
                trends = detect_query_trends(
                    session,
                    f"sc-domain:{domain}",
                    page_path=page_path,
                    days=days,
                    limit=query_limit,
                )
                report["query_intents"] = classify_page_intents(
                    session,
                    f"sc-domain:{domain}",
                    insight_start,
                    insight_end,
                    page_path=page_path,
                    limit=query_limit,
                )
                report["trends"] = {
                    "rising": [t for t in trends if t["trend"] == "rising"],
                    "declining": [t for t in trends if t["trend"] == "declining"],
                }
                report["green_keywords"] = find_green_keywords(
                    session, f"sc-domain:{domain}", page_path, content, days=days
                )

            if page_issues or missing_queries:
                issues[article.url or article.file_path] = {
                    "file_path": article.file_path,
                    "issues": page_issues,
                    "data": {
                        "missing_queries": missing_queries,
                        "keyword_placement": report.get("keyword_placement", {}),
                    },
                }
            page_reports.append(report)
        except Exception as e:
            print(f"Error analyzing {article.file_path}: {e}")

    duplicate_titles = find_duplicate_metadata(session, "title", website_id)
    duplicate_descriptions = find_duplicate_metadata(session, "description", website_id)

    return {
        "total_pages": len(articles),
        "pages_analyzed": len(page_reports),
        "page_reports": page_reports,
        "duplicate_titles": duplicate_titles,
        "duplicate_descriptions": duplicate_descriptions,
        "issues": issues,
        "summary": {
            "total_issues": sum(len(v["issues"]) for v in issues.values()),
            "pages_with_issues": len(issues),
            "duplicate_titles_count": len(duplicate_titles),
            "duplicate_descriptions_count": len(duplicate_descriptions),
        },
    }


In [23]:
# | export
def print_issues_report(report: dict  # Result from `generate_seo_report`
                        ) -> None:
    "Print SEO issues sorted from most issues to fewest."
    issues, summary = report["issues"], report["summary"]
    print(f"Pages analysed   : {report['pages_analyzed']} / {report['total_pages']}")
    print(f"Pages with issues: {summary['pages_with_issues']}")
    print(f"Total issue count: {summary['total_issues']}")
    print(f"Duplicate titles : {summary['duplicate_titles_count']}")
    print(f"Duplicate descs  : {summary['duplicate_descriptions_count']}")
    print("=" * 70)
    for url, data in sorted(issues.items(), key=lambda x: len(x[1]["issues"]), reverse=True):
        count = len(data["issues"])
        print(f"\n[{count} issue{'s' if count != 1 else ''}] {url}")
        for i, issue in enumerate(data["issues"], 1): print(f"  {i}. {issue}")
        if missing := data.get("data", {}).get("missing_queries", []):
            print(f"  Missing queries ({len(missing)}):")
            for q in missing:
                print(
                    f"      - {q['query']} (clicks: {q['total_clicks']}, impressions: {q['total_impressions']}, pos: {q['avg_position']:.1f})")


In [24]:
#| hide
#| eval: false
from seo_rat.content_parser import get_markdown_files
from seo_rat.article import get_articles_by_website
from seo_rat.seo_content_analysis import find_cannibalized


In [25]:
#| hide
#| eval: false
articles = get_articles_by_website(session, website_id=1)
print(len(articles))


87


In [26]:
#| hide
#| eval: false
start, end = get_date_range("last_days", days=200)
# find_cannibalized(session, 8, "sc-domain:gpuvec.com", start, end)



In [27]:
# | hide
# | eval: false
report = generate_seo_report(
    session,
    website_id=1,
    domain="awazly.com",
    is_quarto=False,
    title_is_h1=True,
    desc_field="description",
    date_field="date",
    fetch_base_url="http://localhost:4321/",
)
print(f"Total pages: {report['total_pages']}")
print(f"Issues found: {report['summary']['total_issues']}")


Total pages: 87
Issues found: 362


In [28]:
# | hide
#| eval: false
report


{'total_pages': 87,
 'pages_analyzed': 87,
 'page_reports': [{'file_path': '/home/kobo/Desktop/astro_hacking/awazl/src/content/post/معلم_دهانات_الدوادمي.md',
   'title_check': {'length': 44, 'status': 'too_short', 'min': 50, 'max': 60},
   'description_check': {'length': 0,
    'status': 'missing',
    'min': 150,
    'max': 160},
   'content_check': {'word_count': 2118, 'is_sufficient': True},
   'h1_check': {'h1_count': 1, 'has_single_h1': True, 'h1_source': 'title'},
   'missing_alts': [],
   'link_analysis': {'total_links': 5,
    'internal_count': 5,
    'external_count': 0,
    'internal_links': ['https://awazly.com/trkyb-aybwksy-almdynh-almnwrh',
     'https://awazly.com/trkyb-aybwksy-bmkh',
     'https://awazly.com/trkyb-aybwksy-jdh',
     'https://awazly.com/shrkh-azl-aybwksy-ardhyat-balryadh',
     'https://awazly.com/malm-dhanat-balryadh'],
    'external_links': []},
   'keyword_placement': {'in_h1': False,
    'in_url': False,
    'keyword_in_meta': {'in_title': False, 'in_

In [29]:
#| hide
#| eval: false
print_issues_report(report)


Pages analysed   : 87 / 87
Pages with issues: 87
Total issue count: 362
Duplicate titles : 3
Duplicate descs  : 0

[7 issues] https://awazly.com/malm-dhanat-aldwadmy
  1. No external links
  2. Title too_short (44 chars, range 50-60)
  3. Description missing (0 chars, range 150-160)
  4. Focus keyword not in H1
  5. Focus keyword not in description
  6. Focus keyword not in title
  7. Focus keyword not in first 100 words
  Missing queries (6):
      - طريقة تنظيف سيراميك الباركيه من البويه (clicks: 0, impressions: 25, pos: 75.2)
      - تصميم بوثات بالدوادمي (clicks: 0, impressions: 4, pos: 20.2)
      - شركة عزل مائى بالدوادمي (clicks: 0, impressions: 2, pos: 101.0)
      - بويه ارضيه على شكل سيراميك (clicks: 0, impressions: 2, pos: 87.5)
      - شغل دهانات ترخيم (clicks: 0, impressions: 1, pos: 84.0)
      - بوية ستيكو (clicks: 0, impressions: 1, pos: 47.0)

[7 issues] https://awazly.com/ghsyl-knb-baldwadmy
  1. No external links
  2. Title too_long (79 chars, range 50-60)
  3. Descr

In [ ]:
#| hide
#| eval: false
report['issues']['https://emdadelgaz.com/ahwadh-sbahh-bnzam-alawfr-flw']